# Week 22, Day 5: Reading documentation

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w022d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import inspect
print(inspect.signature(sorted))
print(inspect.signature(round))

In [ ]:
words = "a,b,,c"
print(words.split(","))
print(words.split(",", 1))
print("  two   spaces  ".split())
print("  two   spaces  ".split(" "))

In [ ]:
import inspect, statistics

print(statistics.median.__doc__.splitlines()[0])
print(inspect.signature(statistics.median))
print(inspect.signature(print))
print([name for name in dir(str) if not name.startswith("_")][:10])

In [ ]:
import doctest, math

def split_bill(total, people, tip=0.0):
    """Share a bill equally, rounding each share up to the penny.

    >>> split_bill(100, 3)
    33.34
    >>> split_bill(10, 4, tip=0.2)
    3.0
    """
    if people < 1:
        raise ValueError("people must be at least 1")
    return math.ceil(total * (1 + tip) * 100 / people) / 100

result = doctest.run_docstring_examples(split_bill, {"split_bill": split_bill, "math": math}, verbose=False)
print("all examples passed" if result is None else result)

> **Key idea.** Know which kind of documentation you need: tutorial, how-to, reference or explanation. Read a reference entry signature first (`/` means positional-only, `*` keyword-only), then the first sentence, special cases, errors and version notes, then try it. Use `help`, `__doc__` and `inspect.signature` inside Python. Write docstrings with a summary line, arguments, return value, errors and a `>>>` example, and check the examples with `doctest`.

### Exercise: Read a signature

Write `describe(func)` returning a dictionary from each parameter name of `func` to one of `"positional-only"`, `"either"`, `"keyword-only"`, `"*args"` or `"**kwargs"`, using `inspect.signature(func).parameters` and each parameter's `.kind` (compare with `inspect.Parameter.POSITIONAL_ONLY`, `POSITIONAL_OR_KEYWORD`, `KEYWORD_ONLY`, `VAR_POSITIONAL`, `VAR_KEYWORD`). Then `required(func)`: the names of the parameters with no default (excluding `*args` and `**kwargs`), in order.

In [ ]:
import inspect

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    def f(a, b, /, c, d=1, *rest, e, g=2, **extra):\n        pass\n    assert describe(f) == {\"a\": \"positional-only\", \"b\": \"positional-only\", \"c\": \"either\", \"d\": \"either\", \"rest\": \"*args\", \"e\": \"keyword-only\", \"g\": \"keyword-only\", \"extra\": \"**kwargs\"}\n    assert required(f) == [\"a\", \"b\", \"c\", \"e\"]\n    assert describe(sorted)[\"iterable\"] == \"positional-only\" and describe(sorted)[\"key\"] == \"keyword-only\"")

### Exercise: Look it up

Use the documentation (or `help`, or trying it at the prompt) to fill these in. Each is something people often get wrong.

In [ ]:
a = ___   # "a-b-c".split("-", 1)
b = ___   # round(0.125, 2)  (read what the docs say about rounding halves)
c = ___   # list(range(10, 0, -3))
d = ___   # "Hello".center(9, "*")
e = ___   # sorted(["b", "A", "c"], key=str.lower, reverse=True)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [\"a\", \"b-c\"] and b == round(0.125, 2) and c == [10, 7, 4, 1]\n    assert d == \"**Hello**\" and e == [\"c\", \"b\", \"A\"]")

### Exercise: Write the docstring

Add a Google-style docstring to `bmi` with a one-line summary, an `Args:` section naming both arguments, a `Returns:` section, a `Raises:` section, and **two** `>>>` examples that `doctest` passes: one for a weight of 70 kg and height of 1.75 m (giving `22.9`), one of your own. The checks run your examples with `doctest`.

In [ ]:
def bmi(weight_kg, height_m):
    if height_m <= 0:
        raise ValueError("height must be positive")
    return round(weight_kg / height_m ** 2, 1)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_doc():\n    \"documented and checked\"\n    import doctest\n    doc = bmi.__doc__ or \"\"\n    for section in (\"Args:\", \"Returns:\", \"Raises:\", \"weight_kg\", \"height_m\"):\n        assert section in doc, f\"the docstring needs {section}\"\n    assert doc.count(\">>>\") >= 2 and \"bmi(70, 1.75)\" in doc\n    finder = doctest.DocTestFinder()\n    runner = doctest.DocTestRunner(verbose=False)\n    for t in finder.find(bmi, \"bmi\", globs={\"bmi\": bmi}):\n        runner.run(t)\n    assert runner.failures == 0 and runner.tries >= 2, \"an example in the docstring fails\"")

### Exercise: Documentation coverage

Write `undocumented(source)` returning a sorted list of the names of top-level functions (`ast.FunctionDef` directly in the module's body) that have **no** docstring (use `ast.get_docstring(node)`), and `coverage(source)` returning the percentage of top-level functions with a docstring, rounded to a whole number (100 if there are none).

In [ ]:
import ast

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    src = '''def a():\n    \"\"\"Does a.\"\"\"\ndef b():\n    pass\ndef c():\n    \"\"\"Does c.\"\"\"\n    def inner():\n        pass\ndef d(x):\n    return x\n'''\n    assert undocumented(src) == [\"b\", \"d\"]\n    assert coverage(src) == 50 and coverage(\"x = 1\") == 100")

### Exercise: Version notes

Python's docs mark changes with lines like `Added in version 3.10.` and `Changed in version 3.12: ...` (older pages say `New in version 3.8.`). Write `added_in(doc_text)` returning the version (as a tuple, like `(3, 10)`) from the first `Added in version` or `New in version` note, or `None`, and `usable(doc_text, my_version)` returning `True` if a feature is available in `my_version` (a tuple such as `(3, 9)`), treating "no note" as available.

In [ ]:
import re

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    t1 = \"zip(*iterables, strict=False)\\n...\\nChanged in version 3.10: Added the strict argument.\"\n    t2 = \"math.lcm(*integers)\\nReturn the least common multiple.\\nAdded in version 3.9.\"\n    t3 = \"statistics.correlation(x, y)\\nNew in version 3.10.\"\n    assert added_in(t1) is None and added_in(t2) == (3, 9) and added_in(t3) == (3, 10)\n    assert usable(t2, (3, 9)) and not usable(t3, (3, 9)) and usable(t1, (3, 6))")

### Bug hunt: An example that lies

This docstring's example says `average([1, 2])` gives `1.5`, and it does, but the second example is wrong: the function was changed to round to whole numbers and the docs weren't updated. `doctest` reports the failure. Fix the **docstring** (not the code) so the program prints `examples pass`.

In [ ]:
import doctest

def average(xs):
    """The mean, rounded to a whole number.

    >>> average([1, 2, 4])
    2.3333333333333335
    """
    return round(sum(xs) / len(xs))

runner = doctest.DocTestRunner(verbose=False)
for t in doctest.DocTestFinder().find(average, "average", globs={"average": average}):
    runner.run(t, out=lambda s: None)
print("examples pass" if runner.failures == 0 else "an example fails")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"examples pass\")\n    assert \"return round(sum(xs) / len(xs))\" in source")

### Bug hunt: Read the signature

This crashes with `TypeError: sorted expected 1 argument, got 3`. The signature is `sorted(iterable, /, *, key=None, reverse=False)`. Fix the call so it prints `['apple', 'pear', 'fig']` (longest first).

In [ ]:
words = ["fig", "apple", "pear"]
print(sorted(words, len, True))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"['apple', 'pear', 'fig']\")")

### Puzzle: A documented function

Arrange the lines into a function with a proper docstring and a working example. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know.

In [ ]:
a = ___   # in a signature, parameters after this symbol must be passed by name
b = ___   # the module that runs the >>> examples in docstrings
c = ___   # the kind of documentation for looking up exact details (one word)
d = ___   # True or False: answers from Q&A sites and AI assistants are always up to date

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"*\" and b == \"doctest\" and c == \"reference\" and d is False")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: generate API documentation

Tools like Sphinx and MkDocs build documentation websites from docstrings. Write a small version: `api_docs(source, module_name)` that reads a module's source with `ast` and returns Markdown text:

- a heading `# module_name`, then the module's docstring summary (its first line) if it has one, then a blank line;
- for each top-level **public** function (not starting with `_`), in source order: a heading `## name(params)` where params are the parameter names separated by `, ` (with defaults written `name=default` using `ast.unparse` on the default), then the first line of its docstring or `*Undocumented.*`, then a blank line.

The result's lines are joined with `\n` and end with exactly one `\n`.

In [ ]:
import ast

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SRC = '''\"\"\"Tools for splitting bills.\n\nLonger text that isn't shown.\n\"\"\"\nimport math\n\ndef split_bill(total, people, tip=0.0):\n    \"\"\"Share a bill equally, rounding up to the penny.\"\"\"\n    return math.ceil(total * (1 + tip) * 100 / people) / 100\n\ndef _helper():\n    \"\"\"Private.\"\"\"\n\ndef receipt(items, currency=\"GBP\"):\n    return items\n'''\ndef test_values():\n    \"markdown\"\n    assert api_docs(SRC, \"bills\") == (\n        \"# bills\\n\"\n        \"Tools for splitting bills.\\n\"\n        \"\\n\"\n        \"## split_bill(total, people, tip=0.0)\\n\"\n        \"Share a bill equally, rounding up to the penny.\\n\"\n        \"\\n\"\n        \"## receipt(items, currency='GBP')\\n\"\n        \"*Undocumented.*\\n\"\n    )\n    assert api_docs(\"x = 1\\n\", \"empty\") == \"# empty\\n\"")

## Warm-up

`sorted(iterable, /, *, key=None, reverse=False)`: the `/` and `*` aren't decoration. They say exactly how each argument may be passed, which is why `sorted([3, 1], reverse=True)` works and `sorted([3, 1], None, True)` doesn't. Learning to read signatures like this is half of reading documentation.

## Four kinds of documentation

Good documentation comes in four kinds, each for a different moment. (This way of dividing it, called Diátaxis, was described by Daniele Procida, and Python's own docs broadly follow it.)

| Kind | For when you're... | In Python's docs |
|---|---|---|
| **Tutorial** | learning, step by step, from nothing | "The Python Tutorial" |
| **How-to guide** | trying to get one task done | "Python HOWTOs": sorting, logging, regular expressions... |
| **Reference** | looking up exact details of one thing | "The Python Standard Library", "Language Reference" |
| **Explanation** | trying to understand why | the "what's new" pages, PEPs, design discussions |

Most of the time you'll be in the **reference**: you know roughly what you want, and need the exact details. The standard library reference (docs.python.org/3/library) has a page for every module, with every function, its signature, what it does, what it returns, what it raises, and examples.

Two tips: make sure the version selector at the top of the page matches your Python, and use the **glossary** (docs.python.org/3/glossary.html) when a word like "iterable" or "hashable" is unclear.

## Reading a reference entry

Here's how the docs describe `str.split` (lightly shortened):

```
str.split(sep=None, maxsplit=-1)
    Return a list of the words in the string, using sep as the delimiter string.
    If maxsplit is given, at most maxsplit splits are done (thus, the list will
    have at most maxsplit+1 elements). If maxsplit is not specified or -1, then
    there is no limit on the number of splits.
    If sep is not specified or is None, runs of consecutive whitespace are
    regarded as a single separator...
```

Read it in this order:

1. **The signature:** the names of the parameters, and their **defaults** (`sep=None` means you can leave it out).
2. **The first sentence:** what it returns.
3. **Each parameter's special cases:** here, `None` behaves quite differently from `" "`.
4. **What it raises**, if it says.
5. **Version notes:** "Added in version 3.9" or "Changed in version 3.12" tell you whether your Python has it.
6. **The example**, then try it yourself in a Python session. The docs plus thirty seconds at the `>>>` prompt beat guessing.

**Signature symbols:**

| In a signature | Means | Example |
|---|---|---|
| `name` | can be passed by position or by name | `split(",")` or `split(sep=",")` |
| `name=default` | optional, with that default | `maxsplit=-1` |
| `/` | everything **before** it is positional-only | `sorted(iterable, /, ...)`: you can't write `iterable=` |
| `*` | everything **after** it is keyword-only | `sorted(..., *, key=None)`: you must write `key=` |
| `*args` | any number of extra positional arguments | `print(*objects, ...)` |
| `**kwargs` | any number of extra keyword arguments | `dict(**kwargs)` |
| `[, x]` | (older style) an optional part | `range(start, stop[, step])` |

The last two lines are exactly the special case the docs warn about.

## Help from inside Python

Without leaving Python:

- `help(thing)` shows the full documentation (in a terminal it opens a pager: press `q` to leave).
- `thing.__doc__` is the docstring itself, as a string.
- `inspect.signature(func)` gives the signature, and you can look at each parameter.
- `dir(thing)` lists what's inside a module or object.
- In VS Code, hovering over a name shows the same information.

For third-party packages, the package's own documentation site (linked from its PyPI page) is the reference; its **README** is usually a quick tutorial, and its **changelog** lists what changed in each version, which matters when upgrading.

Sometimes the fastest documentation is the **source code**. For modules written in Python, `inspect.getsource(module.function)` shows it, and every standard library module's source is linked from the top of its documentation page.

## Writing docstrings

Your own functions deserve documentation too. A **docstring** is a string written as the first statement of a function, class or module. A common layout (often called "Google style"):

```
def split_bill(total, people, tip=0.0):
    """Share a bill equally, rounding each share up to the penny.

    Args:
        total: the bill in pounds.
        people: how many people are paying (at least 1).
        tip: the tip as a fraction, e.g. 0.1 for 10%.

    Returns:
        Each person's share in pounds.

    Raises:
        ValueError: if people is less than 1.

    >>> split_bill(100, 3)
    33.34
    """
```

The first line is a one-sentence summary; then what each argument is, what's returned, and what can go wrong. The lines starting `>>>` are an **example**, written exactly as an interactive session would show it.

Those examples can be **run as tests** with the `doctest` module: it finds every `>>>` line, runs it, and checks the result matches the line below. That way the examples in your documentation can never quietly go out of date:

(`run_docstring_examples` prints nothing when every example passes, and prints the details of any that fail. On your own computer, `python -m doctest -v yourfile.py` checks a whole file.)

## Finding answers

1. **Read the error message** carefully, from the bottom up (Week 11). It often says exactly what's wrong.
2. **Check the official docs** for the function or module involved.
3. **Search** with the exact error message (without your own file names) plus the library's name.
4. **Weigh what you find.** Answers on Q&A sites and from AI assistants can be out of date or wrong. Check the date, the Python version, and whether the explanation makes sense, and test it before trusting it.
5. **Ask well.** Make a **minimal reproducible example**: the smallest complete program that shows the problem, with the exact error and the Python version. Making one solves the problem by itself surprisingly often.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick a standard library module you haven't used (try `textwrap`, `itertools` or `pathlib`), read its documentation page for ten minutes, and use two functions from it in a small script. Tomorrow is the Forge: you'll set up a complete, professional project from scratch.